In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q openai psycopg2-binary pgvector pypdf python-dotenv beautifulsoup4 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 31.9 MB/s eta 0:00:00


In [3]:
import os
import openai
import psycopg2
from pgvector.psycopg2 import register_vector
from pypdf import PdfReader
from google.colab import userdata
import requests
from bs4 import BeautifulSoup

In [20]:
host = userdata.get('SUPABASE_HOST')
password = userdata.get('SUPABASE_DB_PASSWORD')

print("Host loaded:", bool(host))
print("Password loaded:", bool(password))

Host loaded: True
Password loaded: True


In [21]:
project_ref = "cpzayxaejvktgerjcmpx"

try:
    conn = psycopg2.connect(
        host=host,
        port=5432,
        database="postgres",
        user=f"postgres.{project_ref}",
        password=password
    )
    cursor = conn.cursor()
    cursor.execute("SELECT 1;")
    result = cursor.fetchone()
    print("Successfully connected to Supabase PostgreSQL database!")
    print("Test query result:", result)
    cursor.close()
    conn.close()
except Exception as e:
    print("An error occurred while connecting to the database:", e)

Successfully connected to Supabase PostgreSQL database!
Test query result: (1,)


In [22]:
import requests
from bs4 import BeautifulSoup

url = "https://nigeriadriverslicence.frsc.gov.ng/faq"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

try:
    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')

    faq_data = []

    # In FRSC's page structure, FAQs are organized as panels
    # The question is in a class like 'panel-heading' or 'panel-title' inside an anchor <a>
    # The answer is in 'panel-collapse' inside a class like 'panel-body'
    panels = soup.find_all(class_="panel")

    for panel in panels:
        header_el = panel.find(class_="panel-heading")
        body_el = panel.find(class_="panel-collapse")

        if header_el and body_el:
            # Extract question text from the anchor inside the header
            question = header_el.get_text(strip=True)
            # Extract answer text from the body
            answer = body_el.get_text(strip=True)

            if question and answer:
                # Avoid duplicates
                if {"question": question, "answer": answer} not in faq_data:
                    faq_data.append({"question": question, "answer": answer})

    # If the standard panel structure is empty, look for accordion-group
    if not faq_data:
        accordion_groups = soup.find_all(class_="accordion-group")
        for group in accordion_groups:
            header_el = group.find(class_="accordion-heading")
            body_el = group.find(class_="accordion-body")
            if header_el and body_el:
                question = header_el.get_text(strip=True)
                answer = body_el.get_text(strip=True)
                if question and answer and {"question": question, "answer": answer} not in faq_data:
                    faq_data.append({"question": question, "answer": answer})

    print(f"Successfully parsed {len(faq_data)} FAQ items!")
    for idx, item in enumerate(faq_data[:5]):
        print(f"\nQ{idx+1}: {item['question']}")
        print(f"A{idx+1}: {item['answer']}")

except Exception as e:
    print("An error occurred while scraping the FAQ page:", e)

Successfully parsed 29 FAQ items!

Q1: How much will the New Licence Cost me?
A1: Licence ClassValidity3 YEARS5 YEARSClass A₦7,000₦11,000Other Classes₦15,000₦21,000Please note the following:1. There might be other charges dependent on the payment channel selected.2. Where applicant chooses to apply for multiple classes, each class will be charged separately.

Q2: How do I pay for the New Licence?
A2: Payment for the new licence can either be made online on the application Portal atnigeriadriverslicence.frsc.gov.ngor at any of the participating banks.

Q3: I just renewed my licence. Can I continue to use it until it expires?
A3: Yes.

Q4: I am a fresh applicant. What do I need to do to obtain the new Driver’s Licence?
A4: 1. Visit an accredited driving school and complete the mandatory drivers training.2. Obtain a driving school certificate number from the accredited driving school to be used to initiate a fresh application.3. Access thenigeriadriverslicence.frsc.gov.ng, click on DL App

In [23]:
import requests
from bs4 import BeautifulSoup

# We will crawl the main landing pages/information sections for processes
urls = {
    "new_licence_info": "https://nigeriadriverslicence.frsc.gov.ng/",
    # Note: If there are specific process page links, we can add them here.
}

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

scraped_processes = []

for name, target_url in urls.items():
    try:
        response = requests.get(target_url, headers=headers, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # Let's extract key text sections, instructions, steps, and requirement listings
        # Most portal instructions sit inside container divs, lists, or card bodies.
        info_sections = []

        # Search for text elements containing keywords like 'how to', 'requirement', 'process', 'step'
        for elem in soup.find_all(['div', 'section', 'ul', 'ol', 'p']):
            text = elem.get_text(" ", strip=True)
            # Filter for meaningful segments containing valuable application steps
            if any(keyword in text.lower() for keyword in ["require", "prerequisite", "step", "process", "guideline", "renew", "reissue"]):
                if len(text) > 100 and text not in info_sections:
                    # Keep the structure somewhat clean
                    info_sections.append(text)

        clean_text = "\n\n".join(info_sections[:10]) # Get the top relevant sections

        scraped_processes.append({
            "source": name,
            "url": target_url,
            "content": clean_text
        })

        print(f"Successfully parsed content from {name}!")
        print(clean_text[:500] + "...\n")

    except Exception as e:
        print(f"Failed to extract information from {name}: {e}")

Successfully parsed content from new_licence_info!
122 (FRSC Emergency Toll Free) | 0700-2255-3772 0807-7690-362 Sign In Home DL Application New Driver's Licence Motor Vehicle Motor Cycle/Tricycle Renewal of Driver's Licence Re-Issue of Driver's Licence Edit Application Acknowledgment Slip Track DL Application Status FAQ Support Contact Accredited Driving Schools Capture Center Make Payment for a fail Payment Apply New Renew Reissue Apply for new driver's licence Use this option if you are applying for a licence for the first time. Prerequisite ...



In [24]:
import requests
from pypdf import PdfReader
import io

pdf_url = "https://frsc.gov.ng/wp-content/uploads/2025/04/DRC-COMPENDIUM-2025.pdf"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

try:
    print(f"Downloading PDF from: {pdf_url}...")
    response = requests.get(pdf_url, headers=headers, timeout=30)
    response.raise_for_status()

    # Load bytes directly into PyPDF
    pdf_file = io.BytesIO(response.content)
    reader = PdfReader(pdf_file)

    pdf_text_chunks = []
    for idx, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and len(text.strip()) > 50:
            pdf_text_chunks.append({
                "page": idx + 1,
                "content": text.strip()
            })

    print(f"Successfully downloaded and extracted {len(pdf_text_chunks)} pages of text from the PDF!")
    if pdf_text_chunks:
        print("\n--- Sample Text from Page 1 ---")
        print(pdf_text_chunks[0]["content"][:800] + "...")

except Exception as e:
    print("An error occurred while downloading or reading the PDF:", e)

Successfully downloaded and extracted 110 pages of text from the PDF!

--- Sample Text from Page 1 ---
2 
 
 
 
1. HIGHWAY CODE LITERACY 
1.1 ROAD SIGNS 
Road traffic signs are structural designs erected along the roadsides for the 
purpose of directing, warning and informing the motoring public and pedestrians of 
road features ahead to guide their decisions. 
A good knowledge of road traffic signs is compulsory for all drivers, as these are 
the basic communication means with the road that guarantees safe motoring. 
These regulate and guide the decisions of drivers well ahead of an y feature they 
would come across on the road. 
The traffic signs are erect, while markings are done on the road pavement, all 
serving the same purpose. 
Road signs are basically of three categories 
 Warning or Danger signs 
 Regulatory signs 
 Informative signs 
Of the three categories of road signs, the ...


In [25]:
from openai import OpenAI

# 1. Load OpenAI API key from secrets
try:
    openai_api_key = userdata.get('OPENAI_API_KEY')
    client = OpenAI(api_key=openai_api_key)
    print("OpenAI client successfully initialized!")
except Exception as e:
    print("Error loading OPENAI_API_KEY from secrets. Make sure it is added with notebook access:", e)

# 2. Function to generate embeddings
def get_embedding(text, model="text-embedding-3-small"):
    text = text.replace("\n", " ")
    response = client.embeddings.create(input=[text], model=model)
    return response.data[0].embedding

# 3. Connect to Supabase, generate embeddings, and insert data
try:
    conn = psycopg2.connect(
        host=host,
        port=5432,
        database="postgres",
        user=f"postgres.{project_ref}",
        password=password
    )
    cursor = conn.cursor()

    # Register pgvector type
    register_vector(conn)

    inserted_count = 0

    for item in faq_data:
        q = item['question']
        a = item['answer']

        # Prepare structured text to embed
        combined_content = f"Question: {q}\nAnswer: {a}"

        # Generate 1536-dimensional embedding
        embedding = get_embedding(combined_content)

        # Insert into our documents table
        cursor.execute("""
            INSERT INTO documents (title, agency, service, context, source_url, embedding)
            VALUES (%s, %s, %s, %s, %s, %s);
        """, (
            q,
            "Federal Road Safety Corps (FRSC)",
            "Driver's Licence Acquisition & Renewal",
            combined_content,
            "https://nigeriadriverslicence.frsc.gov.ng/faq",
            embedding
        ))
        inserted_count += 1

    conn.commit()
    print(f"Successfully embedded and inserted {inserted_count} FAQ documents into Supabase!")

    cursor.close()
    conn.close()
except Exception as e:
    print("An error occurred during embedding or database insertion:", e)

OpenAI client successfully initialized!
Successfully embedded and inserted 29 FAQ documents into Supabase!


In [26]:
from openai import OpenAI
import math

# 1. Initialize OpenAI client safely
try:
    openai_api_key = userdata.get('OPENAI_API_KEY')
    client = OpenAI(api_key=openai_api_key)
    print("OpenAI client successfully initialized!")
except Exception as e:
    print("Error loading OPENAI_API_KEY from secrets:", e)

# 2. Define text chunker function
def chunk_text(text, max_chars=1200, overlap=200):
    """Splits a text string into overlapping chunks for cleaner search indexing."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chars
        chunks.append(text[start:end])
        start += (max_chars - overlap)
    return chunks

# 3. Compile all content sources
documents_to_embed = []

# A. Add FAQs
for item in faq_data:
    q = item['question']
    a = item['answer']
    combined_content = f"Question: {q}\nAnswer: {a}"
    documents_to_embed.append({
        "title": f"FAQ: {q[:100]}",
        "content": combined_content,
        "source_url": "https://nigeriadriverslicence.frsc.gov.ng/faq"
    })

# B. Add Webpage Guide text
for proc in scraped_processes:
    chunks = chunk_text(proc['content'])
    for idx, chunk in enumerate(chunks):
        documents_to_embed.append({
            "title": f"Web Portal Guide - Part {idx+1}",
            "content": chunk,
            "source_url": proc['url']
        })

# C. Add Compendium PDF pages
for chunk_info in pdf_text_chunks:
    page_no = chunk_info['page']
    page_text = chunk_info['content']
    # If page content is long, chunk it further
    chunks = chunk_text(page_text)
    for idx, chunk in enumerate(chunks):
        documents_to_embed.append({
            "title": f"Compendium 2025 (Page {page_no}, Part {idx+1})",
            "content": chunk,
            "source_url": pdf_url
        })

print(f"Prepared a total of {len(documents_to_embed)} document chunks to process and embed.")

# 4. Connect to Supabase, calculate embeddings, and batch upload
try:
    conn = psycopg2.connect(
        host=host,
        port=5432,
        database="postgres",
        user=f"postgres.{project_ref}",
        password=password
    )
    cursor = conn.cursor()
    register_vector(conn)

    inserted_count = 0

    for idx, doc in enumerate(documents_to_embed):
        # Get embedding vector
        clean_content = doc["content"].replace("\n", " ")
        try:
            response = client.embeddings.create(
                input=[clean_content],
                model="text-embedding-3-small"
            )
            embedding = response.data[0].embedding

            # Insert row into Supabase db table (using the 'context' column as per your schema)
            cursor.execute("""
                INSERT INTO documents (title, agency, service, context, source_url, embedding)
                VALUES (%s, %s, %s, %s, %s, %s);
            """, (
                doc["title"],
                "Federal Road Safety Corps (FRSC)",
                "Driver's Licence Portal Support",
                doc["content"],
                doc["source_url"],
                embedding
            ))
            inserted_count += 1
        except Exception as embed_err:
            print(f"Skipping chunk index {idx} due to API/Database error: {embed_err}")
            continue

    conn.commit()
    print(f"\nSuccessfully generated embeddings and uploaded {inserted_count} document chunks to your Supabase PostgreSQL Database!")

    cursor.close()
    conn.close()
except Exception as e:
    print("An error occurred connecting to Supabase database:", e)

OpenAI client successfully initialized!
Prepared a total of 295 document chunks to process and embed.

Successfully generated embeddings and uploaded 295 document chunks to your Supabase PostgreSQL Database!


In [27]:
# Define a query to test similarity search and RAG
test_query = "What are the requirements and steps for a fresh applicant to get a new driver's licence?"

try:
    # 1. Initialize DB Connection
    conn = psycopg2.connect(
        host=host,
        port=5432,
        database="postgres",
        user=f"postgres.{project_ref}",
        password=password
    )
    cursor = conn.cursor()
    register_vector(conn)

    # 2. Embed the search query
    print(f"Embedding search query: '{test_query}'...")
    response = client.embeddings.create(
        input=[test_query.replace("\n", " ")],
        model="text-embedding-3-small"
    )
    query_embedding = response.data[0].embedding

    # 3. Perform Cosine Similarity Search (retrieving top 3 matches using the explicit ::vector cast)
    print("\n--- Executing Vector Similarity Search in Supabase ---")
    cursor.execute("""
        SELECT title, context, source_url, (embedding <=> %s::vector) AS distance
        FROM documents
        ORDER BY distance ASC
        LIMIT 3;
    """, (query_embedding,))

    results = cursor.fetchall()
    retrieved_contexts = []

    for idx, (title, context, source_url, distance) in enumerate(results):
        score = 1 - distance
        retrieved_contexts.append(context)
        print(f"\n[Result {idx+1}] Match Score: {score:.4f} | Source: {title}")
        print(f"URL: {source_url}")
        print(f"Snippet: {context[:250]}...")

    # 4. Generate RAG Answer
    print("\n--- Generating RAG Answer via OpenAI ---")
    system_prompt = """You are an expert assistant specialized in Nigeria Driver's Licence processes.
Use the retrieved context fragments below to answer the user's question accurately.
If the answer cannot be found in the context, tell the user politely."""

    combined_context_text = "\n\n".join([f"[Source: {title}]\n{context}" for title, context, source_url, distance in results])

    rag_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Context:\n{combined_context_text}\n\nQuestion: {test_query}"}
        ],
        temperature=0.3
    )

    print("\nAnswer:")
    print(rag_response.choices[0].message.content)

    cursor.close()
    conn.close()

except Exception as e:
    print("An error occurred during search or retrieval:", e)

Embedding search query: 'What are the requirements and steps for a fresh applicant to get a new driver's licence?'...

--- Executing Vector Similarity Search in Supabase ---

[Result 1] Match Score: 0.6778 | Source: FAQ: I am a fresh applicant. What do I need to do to obtain the new Driver’s Licence?
URL: https://nigeriadriverslicence.frsc.gov.ng/faq
Snippet: Question: I am a fresh applicant. What do I need to do to obtain the new Driver’s Licence?
Answer: 1. Visit an accredited driving school and complete the mandatory drivers training.2. Obtain a driving school certificate number from the accredited dri...

[Result 2] Match Score: 0.6778 | Source: I am a fresh applicant. What do I need to do to obtain the new Driver’s Licence?
URL: https://nigeriadriverslicence.frsc.gov.ng/faq
Snippet: Question: I am a fresh applicant. What do I need to do to obtain the new Driver’s Licence?
Answer: 1. Visit an accredited driving school and complete the mandatory drivers training.2. Obtain a driving 

In [28]:
# Define a new query to test another aspect of the RAG system
test_query_2 = "How much does it cost to renew a driver's licence and what are the validity options?"

try:
    # 1. Initialize DB Connection
    conn = psycopg2.connect(
        host=host,
        port=5432,
        database="postgres",
        user=f"postgres.{project_ref}",
        password=password
    )
    cursor = conn.cursor()
    register_vector(conn)

    # 2. Embed the second test query
    print(f"Embedding search query: '{test_query_2}'...")
    response = client.embeddings.create(
        input=[test_query_2.replace("\n", " ")],
        model="text-embedding-3-small"
    )
    query_embedding_2 = response.data[0].embedding

    # 3. Perform Cosine Similarity Search (retrieving top 3 matches)
    print("\n--- Executing Vector Similarity Search in Supabase ---")
    cursor.execute("""
        SELECT title, context, source_url, (embedding <=> %s::vector) AS distance
        FROM documents
        ORDER BY distance ASC
        LIMIT 3;
    """, (query_embedding_2,))

    results = cursor.fetchall()

    for idx, (title, context, source_url, distance) in enumerate(results):
        score = 1 - distance
        print(f"\n[Result {idx+1}] Match Score: {score:.4f} | Source: {title}")
        print(f"URL: {source_url}")
        print(f"Snippet: {context[:250]}...")

    # 4. Generate RAG Answer
    print("\n--- Generating RAG Answer via OpenAI ---")
    system_prompt = """You are an expert assistant specialized in Nigeria Driver's Licence processes.
Use the retrieved context fragments below to answer the user's question accurately.
If the answer cannot be found in the context, tell the user politely."""

    combined_context_text = "\n\n".join([f"[Source: {title}]\n{context}" for title, context, source_url, distance in results])

    rag_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Context:\n{combined_context_text}\n\nQuestion: {test_query_2}"}
        ],
        temperature=0.3
    )

    print("\nAnswer:")
    print(rag_response.choices[0].message.content)

    cursor.close()
    conn.close()

except Exception as e:
    print("An error occurred during search or retrieval:", e)

Embedding search query: 'How much does it cost to renew a driver's licence and what are the validity options?'...

--- Executing Vector Similarity Search in Supabase ---

[Result 1] Match Score: 0.5876 | Source: Compendium 2025 (Page 92, Part 2)
URL: https://frsc.gov.ng/wp-content/uploads/2025/04/DRC-COMPENDIUM-2025.pdf
Snippet: d Driver‘s Licence may apply for and obtain a renewal 
of the Driver‘s Licence at any time within a period of one calendar month 
before the expiration date of the Licence. 
ii. All applicants for renewal of Driver‘s Licence shall undergo a driving t...

[Result 2] Match Score: 0.5876 | Source: Compendium 2025 (Page 92, Part 2)
URL: https://frsc.gov.ng/wp-content/uploads/2025/04/DRC-COMPENDIUM-2025.pdf
Snippet: d Driver‘s Licence may apply for and obtain a renewal 
of the Driver‘s Licence at any time within a period of one calendar month 
before the expiration date of the Licence. 
ii. All applicants for renewal of Driver‘s Licence shall undergo a driving t...


In [29]:
import json

# 1. Define the actual tool function
def retrieve_driver_licence_knowledge(query: str, limit: int = 3):
    """Performs similarity search on the Supabase vector database using the user's query."""
    try:
        conn = psycopg2.connect(
            host=host,
            port=5432,
            database="postgres",
            user=f"postgres.{project_ref}",
            password=password
        )
        cursor = conn.cursor()
        register_vector(conn)

        # Embed the tool query
        response = client.embeddings.create(
            input=[query.replace("\n", " ")],
            model="text-embedding-3-small"
        )
        query_embedding = response.data[0].embedding

        # Execute similarity search
        cursor.execute("""
            SELECT title, context, source_url, (embedding <=> %s::vector) AS distance
            FROM documents
            ORDER BY distance ASC
            LIMIT %s;
        """, (query_embedding, limit))

        rows = cursor.fetchall()
        cursor.close()
        conn.close()

        formatted_results = []
        for title, context, source_url, distance in rows:
            formatted_results.append({
                "source_title": title,
                "content": context,
                "url": source_url,
                "relevance_score": float(1 - distance)
            })
        return formatted_results
    except Exception as e:
        return f"Database retrieval failed: {str(e)}"

# 2. Define the OpenAI Function Tool schema definition
tools = [
    {
        "type": "function",
        "function": {
            "name": "retrieve_driver_licence_knowledge",
            "description": "Use this tool to search the official Nigeria Driver's Licence database for FAQs, procedures, application steps, requirements, renewal pricing, and regulations from the 2025 Compendium.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The specific search term or question to look up in the database."
                    },
                    "limit": {
                        "type": "integer",
                        "description": "Number of matching document snippets to retrieve.",
                        "default": 3
                    }
                },
                "required": ["query"]
            }
        }
    }
]

# 3. Test the agent loop with function calling enabled
test_user_message = "Can you search the database for what happens if my license expired in 2018?"

print("Sending user request to agent loop...")
initial_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": test_user_message}],
    tools=tools,
    tool_choice="auto"
)

# Print decision
tool_calls = initial_response.choices[0].message.tool_calls
if tool_calls:
    print(f"Agent successfully decided to call function: {tool_calls[0].function.name}")
    args = json.loads(tool_calls[0].function.arguments)
    print(f"Arguments passed to tool: {args}")

    # Execute the function dynamically
    tool_output = retrieve_driver_licence_knowledge(query=args["query"])
    print(f"Retrieved {len(tool_output)} matching snippets from database.")
else:
    print("Agent chose not to call any tools.")

Sending user request to agent loop...
Agent successfully decided to call function: retrieve_driver_licence_knowledge
Arguments passed to tool: {'query': 'expired license 2018'}
Retrieved 3 matching snippets from database.


In [30]:
import json

# 1. Define the tool function exactly matching the user's requested signature
def search_government_information(query: str, limit: int = 3):
    """Searches the official Nigeria Driver's Licence database and 2025 Compendium regulations."""
    try:
        conn = psycopg2.connect(
            host=host,
            port=5432,
            database="postgres",
            user=f"postgres.{project_ref}",
            password=password
        )
        cursor = conn.cursor()
        register_vector(conn)

        # Generate embedding for the tool's search query
        response = client.embeddings.create(
            input=[query.replace("\n", " ")],
            model="text-embedding-3-small"
        )
        query_embedding = response.data[0].embedding

        # Execute cosine similarity vector search
        cursor.execute("""
            SELECT title, context, source_url, (embedding <=> %s::vector) AS distance
            FROM documents
            ORDER BY distance ASC
            LIMIT %s;
        """, (query_embedding, limit))

        rows = cursor.fetchall()
        cursor.close()
        conn.close()

        # Format findings to return to the model
        results = []
        for title, context, source_url, distance in rows:
            results.append({
                "source": title,
                "content": context,
                "url": source_url,
                "relevance_score": float(1 - distance)
            })
        return results
    except Exception as e:
        return f"Retrieval failed: {str(e)}"

# 2. Define the schema list with search_government_information
gov_tools = [
    {
        "type": "function",
        "function": {
            "name": "search_government_information",
            "description": "Use this tool to search the official Nigeria Driver's Licence database, FAQs, portal guidelines, and the 2025 official compendium regulations for accurate information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query or keyword phrase to find in the regulations database."
                    }
                },
                "required": ["query"]
            }
        }
    }
]

# 3. Dynamic Conversation / Agent Loop with Tool Execution & Source Citations
def run_driver_licence_agent(user_question: str):
    messages = [
        {
            "role": "system",
            "content": "You are an expert helper for the Nigeria Driver's Licence portal. You must answer questions using the 'search_government_information' tool. Always provide a clear explanation and explicitly cite the source title and URL of the references returned by the tool."
        },
        {
            "role": "user",
            "content": user_question
        }
    ]

    print(f"User Question: {user_question}\n")

    # Step A: Model decides whether to call the tool
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=gov_tools,
        tool_choice="auto"
    )

    response_message = response.choices[0].message
    messages.append(response_message)

    # Step B: Check for tool calls
    if response_message.tool_calls:
        print("--- Tool Call Initiated by Agent ---")
        for tool_call in response_message.tool_calls:
            if tool_call.function.name == "search_government_information":
                tool_args = json.loads(tool_call.function.arguments)
                search_query = tool_args.get("query")
                print(f"Searching database with query: '{search_query}'...")

                # Execute the retrieval function
                retrieved_data = search_government_information(query=search_query)
                print(f"Retrieved {len(retrieved_data)} records.")

                # Feed the tool execution results back to the model
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": "search_government_information",
                    "content": json.dumps(retrieved_data)
                })

        # Step C: Model synthesizes final response incorporating the tool outputs and sources
        final_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )
        print("\n--- Final Agent Response with Sources ---")
        print(final_response.choices[0].message.content)
    else:
        print("--- No Tool Calls Required ---")
        print(response_message.content)

# 4. Run sample query through the finished loop
run_driver_licence_agent("What happens if my license expired in 2018?")

User Question: What happens if my license expired in 2018?

--- Tool Call Initiated by Agent ---
Searching database with query: 'expired driver's license 2018'...
Retrieved 3 records.

--- Final Agent Response with Sources ---
If your driver's license expired in 2018, you will need to initiate a renewal application through the Federal Road Safety Corps (FRSC) Driver's License Portal. The process is straightforward and can typically be done online.

For more detailed guidance, you can visit the FAQ section of the FRSC Driver's License Portal here: [FRSC DL Portal FAQ](https://nigeriadriverslicence.frsc.gov.ng/faq).


In [31]:
# Execute our newly established agent loop with a realistic inquiry to test tool use
run_driver_licence_agent("What is the renewal cost for a Class B licence and how long is it valid?")

User Question: What is the renewal cost for a Class B licence and how long is it valid?

--- Tool Call Initiated by Agent ---
Searching database with query: 'Class B licence renewal cost'...
Retrieved 3 records.
Searching database with query: 'Class B licence validity period'...
Retrieved 3 records.

--- Final Agent Response with Sources ---
The renewal cost for a Class B driver's licence in Nigeria is ₦15,000 for a validity period of 3 years and ₦21,000 for 5 years. 

Here is a summary of the costs:
- **3 Years Validity**: ₦15,000
- **5 Years Validity**: ₦21,000

Please note that there may be additional charges depending on the payment channel selected, and if you apply for multiple classes, each class will be charged separately. 

For more detailed information, you can refer to the source: [FAQ: How much will the New Licence Cost me?](https://nigeriadriverslicence.frsc.gov.ng/faq).


In [32]:
!git config --global user.name
!git config --global user.email

Denise Moemeke
hello.deniseondata@gmail.com
